# Interrupt Loop Debug

Ask -> LLM -> Ask cycle with a mock Slack server.  
Runs end-to-end in a single notebook (server starts in a background thread).

In [ ]:
import asyncio
import threading
from typing import Any

from examples.slack_mock import MockSlackServer, SlackClient
from hypergraph import END, AsyncRunner, Graph, PauseInfo, interrupt, node, route

## Helpers

In [ ]:
def with_receiver(interrupt_node: Any, receiver: Any) -> Any:
    """Attach a receiver coroutine to an InterruptNode."""
    interrupt_node.receiver = receiver
    return interrupt_node


def fake_llm(messages: list[str]) -> str:
    """Placeholder LLM: echoes the latest user message."""
    latest = next((m for m in reversed(messages) if m.startswith("user: ")), "user: <none>")
    return f"assistant draft based on [{latest}]"


async def run_auto_resume_cycle(
    runner: AsyncRunner,
    graph: Graph,
    values: dict[str, Any],
    *,
    max_pauses: int = 30,
) -> Any:
    """Run a cyclic graph, auto-resuming interrupts via node.receiver."""

    def _resolve_node_by_path(root: Graph, node_path: str) -> Any | None:
        parts = node_path.split("/")
        current_graph = root
        current_node: Any | None = None
        for idx, part in enumerate(parts):
            current_node = current_graph.nodes.get(part)
            if current_node is None:
                return None
            if idx < len(parts) - 1:
                nested = getattr(current_node, "graph", None)
                if nested is None:
                    return None
                current_graph = nested
        return current_node

    merged = dict(values)

    for _ in range(max_pauses + 1):
        result = await runner.run(graph, merged, on_internal_override="ignore")
        if not result.paused:
            return result

        pause = result.pause
        if pause is None:
            raise RuntimeError("Paused run returned no PauseInfo.")

        paused_node = _resolve_node_by_path(graph, pause.node_name)
        if paused_node is None:
            raise RuntimeError(f"Paused node '{pause.node_name}' not found.")

        receiver = getattr(paused_node, "receiver", None)
        if receiver is None:
            raise RuntimeError(f"Node '{pause.node_name}' has no receiver.")

        print(f"[demo] paused at '{pause.node_name}', waiting for reply...")
        response = receiver(pause)
        if asyncio.iscoroutine(response):
            response = await response

        if pause.value is not None:
            merged["messages"] = pause.value
        merged[pause.response_key] = response
        print(f"[demo] received -> {pause.response_key}={response!r}")

    raise RuntimeError(f"Exceeded max_pauses={max_pauses}.")

## Node definitions

All functions are portable: `slack` and `question` are injected via `values`, not captured from globals.

In [ ]:
# @interrupt(output_name="user_input")
# def ask_slack(messages: list[str], slack, question) -> None:
#     turn_number = sum(1 for m in messages if m.startswith("assistant: ")) + 1
#     prompt = f"Turn {turn_number}. {question}"
#     if messages:
#         prompt = f"Turn {turn_number}. Prior context:\n" + "\n".join(messages[-4:])
#     slack.post_message(prompt)
#     return None


@interrupt(output_name="user_input")
def ask_slack(messages: list[str], slack) -> None:
    turn_number = sum(1 for m in messages if m.startswith("assistant: ")) + 1
    prompt = f"Turn {turn_number}. Prior context:\n" + "\n".join(messages[-4:])
    slack.post_message(prompt)
    return None


@node(output_name="assistant_text")
def llm_step(messages: list[str]) -> str:
    return fake_llm(messages)


@node(output_name="messages")
def add_user_message(messages: list[str], user_input: str) -> list[str]:
    return [*messages, f"user: {user_input}"]


@node(output_name="messages")
def add_assistant_message(messages: list[str], assistant_text: str) -> list[str]:
    return [*messages, f"assistant: {assistant_text}"]


@route(targets=["ask_user", END])
def should_continue(messages: list[str], max_turns: int) -> str:
    turns = sum(1 for m in messages if m.startswith("assistant: "))
    return END if turns >= max_turns else "ask_user"

## Graph construction

In [ ]:
ask_user_node = Graph(
    [ask_slack, add_user_message],
    edges=[(ask_slack, add_user_message)],
    name="ask_user",
    entrypoint="ask_slack",
).as_node()

llm_node = Graph(
    [llm_step, add_assistant_message],
    edges=[(llm_step, add_assistant_message)],
    name="llm",
    entrypoint="llm_step",
).as_node()

graph = Graph(
    [ask_user_node, llm_node, should_continue],
    edges=[
        (ask_user_node, llm_node),
        (llm_node, should_continue),
        (llm_node, ask_user_node),
    ],
    name="slack_cycle",
    entrypoint="ask_user",
)

In [ ]:
graph.visualize(depth=2)

In [ ]:
# @interrupt(output_name="user_input")
# def ask_slack(messages: list[str], slack, question) -> None:
#     turn_number = sum(1 for m in messages if m.startswith("assistant: ")) + 1
#     prompt = f"Turn {turn_number}. {question}"
#     if messages:
#         prompt = f"Turn {turn_number}. Prior context:\n" + "\n".join(messages[-4:])
#     slack.post_message(prompt)
#     return None


@interrupt(output_name="user_input")
def ask_slack(messages: list[str], slack) -> None:
    turn_number = sum(1 for m in messages if m.startswith("assistant: ")) + 1
    prompt = f"Turn {turn_number}. Prior context:\n" + "\n".join(messages[-4:])
    slack.post_message(prompt)
    return None


@node(output_name="assistant_text")
def llm_step(messages: list[str]) -> str:
    return fake_llm(messages)


@node(output_name="messages")
def add_user_message(messages: list[str], user_input: str) -> list[str]:
    return [*messages, f"user: {user_input}"]


@node(output_name="messages")
def add_assistant_message(messages: list[str], assistant_text: str) -> list[str]:
    return [*messages, f"assistant: {assistant_text}"]


@route(targets=["ask_slack", END])
def should_continue(messages: list[str], max_turns: int) -> str:
    turns = sum(1 for m in messages if m.startswith("assistant: "))
    return END if turns >= max_turns else "ask_slack"

In [ ]:
# should_continue

In [ ]:
graph = Graph(
    [ask_slack, add_user_message, llm_step, add_assistant_message, should_continue],
    edges=[(add_user_message, llm_step), (add_assistant_message, should_continue)],
    #     (ask_slack, add_user_message),
    #     (llm_step, add_assistant_message),
    #     (add_assistant_message, ask_slack),
    #     (add_assistant_message, add_user_message),
    #     # (add_assistant_message, should_continue),
    #     # (should_continue, ask_slack),
    # ],
    name="slack_cycle",
    shared=["messages"],
    entrypoint="ask_slack",
)

In [ ]:
graph.visualize()

## Run

Start mock Slack server in a background thread, pre-queue responses, then execute.

In [ ]:
# Config
SLACK_URL = "http://127.0.0.1:8765"
question = "Should we ship this policy update?"
max_turns = 3

# Start mock server
server = MockSlackServer(port=8765)
threading.Thread(target=server.serve_forever, daemon=True).start()

# Create client
slack = SlackClient(SLACK_URL)

In [ ]:
# Wire the receiver without relying on argparse/global args
def make_receive_slack(slack_client: SlackClient):
    async def receive_slack(_: PauseInfo) -> str:
        return await slack_client.receive_response()

    return receive_slack


with_receiver(ask_slack, make_receive_slack(slack))

# Pre-queue one reply per turn so the demo runs without manual input
for i in range(max_turns):
    slack.queue_response(f"Looks good, iteration {i + 1}")

In [ ]:
graph = graph.bind(messages=[], max_turns=3, slack=slack)

In [ ]:
runner = AsyncRunner()
result = await run_auto_resume_cycle(
    runner,
    graph,
    values={},
)

In [ ]:
result

In [ ]:
print("[final messages]")
for line in result["messages"]:
    print(f"  {line}")

In [ ]:
server.close()